# Agente analista de datos

## Objetivo

Que el/la estudiante implemete un agente inteligente usando la librería *crewai* complementado con un LLM (*llama3.2*) para generar repotes de ventas, almacen y clientes.

# instalación de librerías
!pip install creaw pandas

## Definición de funciones

In [11]:
!pip install pandas


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
# importando pandas
import pandas as pd

In [13]:
# Función para cargar el conjunto de datos ventas
def cargar_ventas():
    df = pd.read_csv('ventas.csv')
    return df

In [14]:
ventas = cargar_ventas()
ventas.head()  # Muestra las primeras filas del DataFrame

,fecha,sucursal,producto,cantidad,precio_unitario,ventas
0,2025-01-01,Norte,Laptop,5,18000,90000
1,2025-01-01,Norte,Tablet,4,7000,28000
2,2025-01-01,Norte,Smartphone,4,12000,48000
3,2025-01-01,Norte,Monitor,5,3500,17500
4,2025-01-01,Norte,Teclado,5,800,4000


In [15]:
# Función para calcular el total de ventas
def total_ventas():
    df = cargar_ventas()
    total = df['ventas'].sum()
    return total

In [16]:
# Función para identificar la lista productos con mayores ventas
def top_productos(n=5):
    df = cargar_ventas()
    productos = (
df.groupby('producto')['ventas']
       .sum()
       .sort_values(ascending=False)
       .head(n)
    )
    return productos

In [17]:
# Función para identificar la lista de sucursales con mayores ventas
def top_sucursales(n=5):
    df = cargar_ventas()
    sucursales = (
        df.groupby('sucursal')['ventas']
        .sum()
        .sort_values(ascending=False)
        .head(n)
    )
    return sucursales

In [29]:
print("Top de ventas:", top_productos())
print("Top de sucursales:", top_sucursales())

Top de ventas: producto
Laptop        46836000
Smartphone    32640000
Tablet        19544000
Smartwatch    13750000
Impresora     11314800
Name: ventas, dtype: int64
Top de sucursales: sucursal
Online      25632150
Oriente     25238600
Poniente    25096900
Centro      24743850
Sur         24537750
Name: ventas, dtype: int64


## Construyendo un agente con CrewAI

In [18]:
# importación de elementos del agente-analista
from crewai.tools import tool
from crewai import Agent, Task, Crew, LLM

In [32]:
# Definición de herramientas
@tool
def calcular_total_ventas():
    """Calcula el total de las ventas"""
    return total_ventas().to_string()

@tool
def obtener_top_productos(n=5):
    """Obtiene la lista de productos con mayores ventas"""
    return top_productos(n).to_string()

@tool
def obtener_top_sucursales(n=5):
    """Obtiene la lista de sucursales con mayores ventas"""
    return top_sucursales(n).to_string()

In [33]:
# Definición del modelo de lenguaje a utilizar
llm = LLM(
    #model="llama3.2",
    model="ollama/llama3.2",
    base_url="http://localhost:11434",
    api_key="ollama"
)

In [34]:
agente_analista = Agent(
    name="Agente Analista",
    role="Eres un agente analista de ventas que ayuda a los usuarios a analizar datos de ventas y proporcionar información sobre el total de ventas, los productos más vendidos y las sucursales con mayores ventas.",
    goal="Ayuda a los usuarios a analizar datos de ventas y proporcionar información sobre el total de ventas, los productos más vendidos y las sucursales con mayores ventas.",
    backstory="Soy un agente analista de ventas en qué te puedo ayudar.",
    description="Un agente que analiza datos de ventas y proporciona información sobre el total de ventas, los productos más vendidos y las sucursales con mayores ventas.",
    tools=[calcular_total_ventas, obtener_top_productos, obtener_top_sucursales],
    llm=llm,
    verbose=True
)

## Prueba de funcionamiento y disponiblidad del LLM

In [ ]:
import requests

def generate_response(prompt, model="llama3.2"):
    try:
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model": model,
                "prompt": prompt,
                "stream": False
            }
        )
        response.raise_for_status() # Lanza un error para códigos de estado HTTP erróneos
        return response.json()["response"]
    except requests.exceptions.RequestException as e:
        return f"Error al conectar con Ollama o generar respuesta: {e}"


# Prueba la generación de respuesta
ollama_response = generate_response("Explica brevemente qué es la inteligencia artificial.")
print("\nRespuesta de Llama 3.2 a través de Ollama:")
print(ollama_response)


Respuesta de Llama 3.2 a través de Ollama:
**Inteligencia Artificial (IA): Una visión general**

La inteligencia artificial es un campo de estudio que se centra en el desarrollo de sistemas y algoritmos que pueden realizar tareas que normalmente requieren la inteligencia humana, como el aprendizaje, la resolución de problemas y la toma de decisiones.

**Definición**

La inteligencia artificial se define como la simulación de la inteligencia humana mediante la creación de sistemas informáticos que puedan:

* Aprender de datos y mejorar su desempeño
* Tomar decisiones basadas en información y reglas
* Resolver problemas complejos y abstractos
* Comprender y generar lenguaje natural

**Tipos de inteligencia artificial**

Existen varios tipos de inteligencia artificial, incluyendo:

* **Inteligencia artificial débil**: se enfoca en resolver problemas específicos y limitados
* **Inteligencia artificial fuerte**: se enfoca en crear sistemas que puedan realizar tareas complejas y abstractas 

In [35]:
# Definición de tareas para el agente analista
tarea_total_ventas = Task(
    description="""
    Analiza los productos mas vendidos 
    usando las herramientas definidas y calcula el
    total de ventas en el conjunto de datos.
    """,
    expected_output="""
    Un informe ejecutivo en el idioma español que incluya 
    el total de ventas, lista de los productos más vendidos
    y las sucursales con mayores ventas. 
    El informe debe ser claro, conciso y fácil de entender
    para un público general.
    """,
    agent=agente_analista
)

In [38]:
# Definición del Crew
crew = Crew(
    name="Crew de Agentes Analistas",
    description="Un equipo de agentes analistas de ventas que trabajan juntos para analizar datos de ventas",
    agents=[agente_analista],
    tasks=[tarea_total_ventas],
    verbose=True
)

In [ ]:
# Ejecutar la tarea del Crew
async def ejecutar_tarea_crew():
    try:
        resultado = await crew.kickoff_async()
        print("Resultado de la tarea del Crew:")
        print(resultado)
    except Exception as e:
        print(f"Error al ejecutar la tarea del Crew: {e}")

await ejecutar_tarea_crew()

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.22                                                                                       │
│  Latest version:  1.15.23                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: Crew de Agentes Analistas                                                                                │
│  ID: 2bd2fe48-e07a-449a-bdd5-dac0adc45114                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│      Analiza los productos mas vendidos                                                                         │
│      usando las herramientas definidas y calcula el                                                             │
│      total de ventas en el conjunto de datos.                                                                   │
│                                                                                                                 │
│  ID: 9154b402-fdcd-46e4-8a86-b602aac15c94                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Eres un agente analista de ventas que ayuda a los usuarios a analizar datos de ventas y proporcionar    │
│  información sobre el total de ventas, los productos más vendidos y las sucursales con mayores ventas.          │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Analiza los productos mas vendidos                                                                         │
│      usando las herramientas definidas y calcula el                                                             │
│      total de ventas en el conjunto de datos.                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: obtener_top_productos                                                                                    │
│  Args: {'n': '10'}                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool obtener_top_productos executed with result: Error executing tool: cannot do positional indexing on Index with these indexers [10] of type str...


╭────────────────────────────────────────────── 🔧 Tool Error (#1) ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Failed                                                                                                    │
│  Tool: obtener_top_productos                                                                                    │
│  Iteration: 1                                                                                                   │
│  Attempt: 0                                                                                                     │
│  Error: cannot do positional indexing on Index with these indexers [10] of type str                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯